In [15]:
import yfinance as yf

In [16]:
import pandas as pd

# Top 10 US stocks by market cap (typical order; use yfinance for live P/E)
# Sources: major indices / market cap rankings (NVDA, AAPL, MSFT, GOOGL, AMZN, META, AVGO, TSLA, BRK-B, LLY)
tickers = ["NVDA", "AAPL", "MSFT", "GOOGL", "AMZN", "META", "AVGO", "TSLA", "BRK-B", "LLY"]
print("Top 10 US stocks by market cap (tickers):", tickers)

Top 10 US stocks by market cap (tickers): ['NVDA', 'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'AVGO', 'TSLA', 'BRK-B', 'LLY']


In [17]:
# Top 10 by P/E ratio (ascending – lowest P/E first), using financials.ratios.pe
from financials.ratios import pe
import yfinance as yf

pe_data = []
for ticker in tickers:
    t = yf.Ticker(ticker)
    info = t.info
    name = info.get("shortName", ticker)
    pe_data.append({
        "Ticker": ticker,
        "Company": name,
        "P/E": pe(ticker),
    })

df_pe = pd.DataFrame(pe_data)
df_pe = df_pe.sort_values(by="P/E", ascending=True, na_position="last")
df_pe

,Ticker,Company,P/E
8,BRK-B,Berkshire Hathaway Inc. New,15.896166
2,MSFT,Microsoft Corporation,25.129618
5,META,"Meta Platforms, Inc.",27.247446
4,AMZN,"Amazon.com, Inc.",27.686628
3,GOOGL,Alphabet Inc.,28.255085
1,AAPL,Apple Inc.,32.377216
0,NVDA,NVIDIA Corporation,45.250000
9,LLY,Eli Lilly and Company,45.256744
6,AVGO,Broadcom Inc.,68.027200
7,TSLA,"Tesla, Inc.",382.972470


In [18]:
# Valuation ratios: P/E, PEG (5yr expected), Price/Sales, EV/EBITDA
from financials.ratios import pe, peg_ratio, price_to_sales, ev_to_ebitda
import yfinance as yf

ratio_data = []
for ticker in tickers:
    t = yf.Ticker(ticker)
    info = t.info
    ratio_data.append({
        "Ticker": ticker,
        "Company": info.get("shortName", ticker),
        "P/E": pe(ticker),
        "PEG Ratio (5yr expected)": peg_ratio(ticker),
        "Price/Sales": price_to_sales(ticker),
        "EV/EBITDA": ev_to_ebitda(ticker),
    })

df_ratios = pd.DataFrame(ratio_data)

# 4 views: P/E ascending, PEG descending, Price/Sales ascending, EV/EBITDA descending
print("1. By P/E (ascending – lowest first)")
display(df_ratios.sort_values(by="P/E", ascending=True, na_position="last")[["Ticker", "Company", "P/E"]])

print("2. By PEG Ratio (5yr expected) (descending)")
display(df_ratios.sort_values(by="PEG Ratio (5yr expected)", ascending=False, na_position="last")[["Ticker", "Company", "PEG Ratio (5yr expected)"]])

print("3. By Price/Sales (ascending)")
display(df_ratios.sort_values(by="Price/Sales", ascending=True, na_position="last")[["Ticker", "Company", "Price/Sales"]])

print("4. By EV/EBITDA (descending)")
display(df_ratios.sort_values(by="EV/EBITDA", ascending=False, na_position="last")[["Ticker", "Company", "EV/EBITDA"]])

1. By P/E (ascending – lowest first)


,Ticker,Company,P/E
8,BRK-B,Berkshire Hathaway Inc. New,15.896166
2,MSFT,Microsoft Corporation,25.129618
5,META,"Meta Platforms, Inc.",27.247446
4,AMZN,"Amazon.com, Inc.",27.686628
3,GOOGL,Alphabet Inc.,28.255085
1,AAPL,Apple Inc.,32.377216
0,NVDA,NVIDIA Corporation,45.250000
9,LLY,Eli Lilly and Company,45.256744
6,AVGO,Broadcom Inc.,68.027200
7,TSLA,"Tesla, Inc.",382.972470


2. By PEG Ratio (5yr expected) (descending)


,Ticker,Company,PEG Ratio (5yr expected)
7,TSLA,"Tesla, Inc.",6.2673
3,GOOGL,Alphabet Inc.,2.3181
1,AAPL,Apple Inc.,2.3096
4,AMZN,"Amazon.com, Inc.",1.8173
2,MSFT,Microsoft Corporation,1.5150
5,META,"Meta Platforms, Inc.",1.1501
6,AVGO,Broadcom Inc.,0.9035
9,LLY,Eli Lilly and Company,0.8894
0,NVDA,NVIDIA Corporation,0.7060
8,BRK-B,Berkshire Hathaway Inc. New,NaN


3. By Price/Sales (ascending)


,Ticker,Company,Price/Sales
8,BRK-B,Berkshire Hathaway Inc. New,2.884443
4,AMZN,"Amazon.com, Inc.",2.976598
5,META,"Meta Platforms, Inc.",8.052774
1,AAPL,Apple Inc.,8.630139
3,GOOGL,Alphabet Inc.,9.180647
2,MSFT,Microsoft Corporation,9.765044
9,LLY,Eli Lilly and Company,14.303918
7,TSLA,"Tesla, Inc.",16.518661
0,NVDA,NVIDIA Corporation,23.783413
6,AVGO,Broadcom Inc.,24.131983


4. By EV/EBITDA (descending)


,Ticker,Company,EV/EBITDA
7,TSLA,"Tesla, Inc.",146.416
0,NVDA,NVIDIA Corporation,38.977
9,LLY,Eli Lilly and Company,30.472
1,AAPL,Apple Inc.,24.714
3,GOOGL,Alphabet Inc.,24.228
2,MSFT,Microsoft Corporation,17.197
5,META,"Meta Platforms, Inc.",15.917
4,AMZN,"Amazon.com, Inc.",15.024
6,AVGO,Broadcom Inc.,4.657
8,BRK-B,Berkshire Hathaway Inc. New,-2.421


In [19]:
# Diluted EPS for top 10 stocks – last 5 years (using financials.eps)
from financials.eps import annual_eps

years = list(range(pd.Timestamp.now().year - 4, pd.Timestamp.now().year))
rows = []
for ticker in tickers:
    row = {"Ticker": ticker}
    for y in years:
        row[y] = annual_eps(ticker, y)
    rows.append(row)

df_diluted_eps = pd.DataFrame(rows).set_index("Ticker")
# Add change 2024 → 2025 and sort by it (descending)
if 2024 in df_diluted_eps.columns and 2025 in df_diluted_eps.columns:
    df_diluted_eps["Change (2024→2025)"] = df_diluted_eps[2025] - df_diluted_eps[2024]
    df_diluted_eps = df_diluted_eps.sort_values(by="Change (2024→2025)", ascending=False, na_position="last")
else:
    last_year = years[-1]
    df_diluted_eps = df_diluted_eps.sort_values(by=last_year, ascending=False, na_position="last")
df_diluted_eps

,2022,2023,2024,2025,Change (2024→2025)
Ticker,,,,,
LLY,6.572215,5.800000,11.710000,22.95,11.24
AVGO,2.653000,3.298000,1.230000,4.77,3.54
GOOGL,4.560000,5.800000,8.040000,10.81,2.77
MSFT,9.650000,9.680000,11.800000,13.64,1.84
NVDA,0.385000,0.174000,1.190000,2.94,1.75
AMZN,-0.270000,2.900000,5.530000,7.17,1.64
AAPL,6.110000,6.130000,6.080000,7.46,1.38
TSLA,3.620000,4.310000,2.040000,1.08,-0.96
META,8.590000,14.870000,23.860000,NaN,NaN


In [20]:
# Quarterly diluted EPS for top 10 stocks – last 3 years (each quarter)
import financials.eps
import importlib
importlib.reload(financials.eps)
from financials.eps import quarterly_eps

years = list(range(pd.Timestamp.now().year - 2, pd.Timestamp.now().year))
quarters = ["Q1", "Q2", "Q3", "Q4"]
cols = [f"{y} {q}" for y in years for q in quarters]

rows = []
for ticker in tickers:
    row = {"Ticker": ticker}
    for y in years:
        for q in quarters:
            row[f"{y} {q}"] = quarterly_eps(ticker, y, q)
    rows.append(row)

df_quarterly_eps = pd.DataFrame(rows).set_index("Ticker")
# Same descending order by last year EPS as annual table
df_quarterly_eps = df_quarterly_eps.reindex(df_diluted_eps.index)
df_quarterly_eps

,2024 Q1,2024 Q2,2024 Q3,2024 Q4,2025 Q1,2025 Q2,2025 Q3,2025 Q4
Ticker,,,,,,,,
LLY,None,NaN,NaN,4.88,3.060000,6.290000,6.21,7.39
AVGO,None,NaN,NaN,0.90,1.140000,1.030000,0.85,1.74
GOOGL,None,NaN,NaN,2.15,2.810000,2.310000,2.87,2.82
MSFT,None,NaN,NaN,3.23,3.460000,3.650000,3.72,5.16
NVDA,None,NaN,NaN,0.78,0.890000,0.760000,1.08,1.30
AMZN,None,NaN,NaN,1.86,1.590000,1.680000,1.95,1.95
AAPL,None,NaN,NaN,2.40,1.650000,1.570000,1.85,2.84
TSLA,None,NaN,NaN,0.66,0.120000,0.330000,0.39,0.24
META,None,NaN,NaN,8.02,6.430000,7.140000,1.05,8.88


In [21]:
# Example: use insider package to inspect NVDA insider activity
from insider import (
    insider_trades_df,
    collapse_form4_trades,
    filter_open_market,
    net_insider_flow,
)

# Fetch raw Form 4 data
trades_df = insider_trades_df("NVDA", limit=50)

# Optional: cleaned, one row per logical trade
collapsed = collapse_form4_trades(trades_df)
print("\nCollapsed trades (head):")
print(collapsed.head())

# Optional: open-market only
open_market = filter_open_market(collapsed)
print("\nOpen‑market only (head):")
print(open_market.head())

# Optional: 90d net flow
print("\nNet insider flow (90d):", net_insider_flow(trades_df, days=90))



Collapsed trades (head):
   ticker        insider                          title filing_date  \
65   NVDA  Kress Colette  EVP & Chief Financial Officer  2026-02-06   
66   NVDA  Kress Colette  EVP & Chief Financial Officer  2026-02-06   
71   NVDA    Puri Ajay K       EVP, Worldwide Field Ops  2026-01-23   
63   NVDA  Kress Colette  EVP & Chief Financial Officer  2026-01-15   
64   NVDA  Kress Colette  EVP & Chief Financial Officer  2026-01-15   

   transaction_date transaction_code direct_or_indirect    shares  \
65       2026-02-04                S                  D   27640.0   
66       2026-02-04                S                  I   20000.0   
71       2026-01-21                S                  I  200000.0   
63       2026-01-13                S                  D   27640.0   
64       2026-01-13                S                  I   20000.0   

           value  shares_owned_after  price_vwap  
65  4.856861e+06            846772.0  175.718571  
66  3.514372e+06             9

In [22]:
# Example: rank by Price/Sales (ascending) from nasdaq100 CSV
from financials.ratios import price_to_sales
from screening import rank_stocks_by_ratio

df = rank_stocks_by_ratio(price_to_sales, csv_path="nasdaq100_symbols.csv", ascending=True)
print(df.head(20))

   ticker     ratio
0    CHTR  0.564875
1   CMCSA  0.929921
2    PYPL  1.136423
3     KHC  1.177005
4    CTSH  1.521704
5    COST  1.612613
6    GEHC  1.775270
7     EXC  2.018476
8    MDLZ  2.095740
9     BKR  2.180798
10   PCAR  2.345800
11    PEP  2.415719
12    KDP  2.493900
13   TMUS  2.780149
14   SBUX  2.834221
15   ROST  2.901846
16   AMZN  2.976598
17    AEP  3.212632
18    XEL  3.290186
19   QCOM  3.349320


In [1]:
# Example: CANSLIM ranking for nasdaq symbols
from screening import rank_stocks_by_canslim

df = rank_stocks_by_canslim(
    csv_path="nasdaq_symbols.csv",
    limit=5000,
    weights={"C": 20, "A": 15, "N": 15, "S": 10, "L": 25, "I": 10, "M": 5},
)
print(df.head(25))

Failed to get ticker 'LAKE' reason: Failed to perform, curl: (28) Connection timed out after 10004 milliseconds. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.

474 Failed downloads:
['INDB', 'JLHL', 'BHST', 'KVACW', 'NVX', 'CPBI', 'RDNT', 'NBIL', 'OSUR', 'DWTX', 'UTWO', 'RTAC', 'IBG', 'KVAC', 'CBK', 'CAFG', 'NOEMU', 'RTYY', 'CLNE', 'CORZZ', 'FMBH', 'LTRX', 'BOLD', 'CDXS', 'GSIB', 'GMM', 'DAAQW', 'BGL', 'MRCY', 'BOTZ', 'AMAL', 'PRTA', 'GIII', 'LEGN', 'NTWOW', 'DRIO', 'DAKT', 'UONEK', 'FFIC', 'NNBR', 'IVSI', 'BAER', 'NSI', 'VCYT', 'SWKS', 'FUND', 'NBTX', 'NIOBW', 'CYCN', 'KRYS', 'ONC', 'JUNS', 'OLOX', 'CGCTW', 'MSBIP', 'NWGL', 'LWLG', 'SNES', 'BMAX', 'ADACU', 'MOOD', 'CHCO', 'LBGJ', 'LMTS', 'SUPX', 'NRDS', 'AXON', 'INSG', 'SCLXW', 'PSCU', 'BTCS', 'AGEM', 'IBIT', 'QQHG', 'ASPS', 'IEF', 'BCARW', 'MYCI', 'LOBO', 'EMPG', 'METD', 'ALRS', 'CDIOW', 'LNZAW', 'GAIA', 'FLEX', 'BGRO', 'IFRX', 'OSS', 'CARG', 'PFIS', 'LYRA', 'TSMX', 'BMRN', 'TJGC', 'CRUS', 'INMB', 'REAI', 

   ticker          C          A           N          S          L           I  \
0    NRDS  27.039242  98.907493   57.942254   2.703107  50.000000   80.450000   
1   AGNCZ   0.803781  51.160763   99.846915   0.511892  50.000000  100.000000   
2    ANSC   0.715260  50.831309   99.964425   0.003085  50.000000  100.000000   
3    DGII   0.119159  49.767720   99.855159   2.484938  50.000000  100.000000   
4    PRAX  50.000000  49.602995   94.489804   1.009637  15.857416   94.457000   
5   MFICL   0.192463  49.897854  100.000000   0.013822  50.000000  100.000000   
6    AKAM   0.248701  50.024877   98.466920   4.771569  50.000000   97.095000   
7   HBANZ   0.100447  49.720681   99.539371   0.724084  50.000000  100.000000   
8    CRUS   0.124196  49.788402   98.464693   2.160714  50.000000  100.000000   
9   CCNEP   0.229475  49.860882   99.098016   0.836088  50.000000  100.000000   
10  FITBM   0.121009  49.778702   99.470485   0.089417  50.000000  100.000000   
11    HST   0.201614  49.917